# 🛠️ PKDD Financial Dataset Cleansing Pipeline
This notebook loads, cleans, standardises, and integrates the eight relational tables in the **PKDD Financial Database**:
- **Relational Tables**: `Account.csv`, `Card.csv`, `Client.csv`, `Disp.csv`, `District.csv`, `Loan.csv`, `Order.csv`
- **Ledger Files**: Data cleansing


In [2]:
# ==========================================
# 1. SETUP & MOUNT GOOGLE DRIVE
# ==========================================
import pandas as pd
import numpy as np
import sqlite3
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define base folder path containing your CSV files
DATA_DIR = '/content/drive/MyDrive/Capstone_Project/'

print("✅ Google Drive mounted successfully!")


Mounted at /content/drive
✅ Google Drive mounted successfully!


## 1. Transaction Ledger Cleansing
Handles missing counterparty attributes, parses date columns, and translates Czech operational codes into standard English terms.


In [3]:
# ==========================================
# 1. TRANSACTIONS CLEANSING & EXPORT
# ==========================================
# Define raw input path and cleaned output path
raw_trans_path = os.path.join(DATA_DIR, 'Transaction.csv')
clean_trans_path = os.path.join(DATA_DIR, 'Transaction_clean.csv')

# Load raw Transaction.csv file and drop completely empty rows
# (Note: If your files are split into parts, use: glob.glob(os.path.join(DATA_DIR, 'Transaction_part_*.csv')))
df_transactions = pd.read_csv(raw_trans_path).dropna(how='all')

# 1. Deduplicate records on unique transaction ID
df_transactions = df_transactions.drop_duplicates(subset=['trans_id'])

# 2. Standardise date column to datetime
df_transactions['date'] = pd.to_datetime(df_transactions['date'], errors='coerce')

# 3. Ensure numeric types for monetary values
df_transactions['amount'] = pd.to_numeric(df_transactions['amount'], errors='coerce')
df_transactions['balance'] = pd.to_numeric(df_transactions['balance'], errors='coerce')

# 4. Map legacy Czech financial operational codes to English
type_map = {
    'PRIJEM': 'Credit (Income)',
    'VYDAJ': 'Debit (Expense)',
    'VYBER': 'Cash Withdrawal'
}
operation_map = {
    'VYBER KE KARTY': 'Credit Card Withdrawal',
    'VKLAD': 'Cash Deposit',
    'PREVOD Z UCTU': 'Remittance from Other Bank',
    'VYBER': 'Cash Withdrawal',
    'PREVOD NA UCET': 'Remittance to Other Bank'
}
k_symbol_map = {
    'POJISTNE': 'Insurance Payment',
    'SIPO': 'Household/Utility Payment',
    'UVER': 'Loan Payment',
    'DUCHOD': 'Pension',
    'SLUZBY': 'Service Charge',
    'SANKC. UROK': 'Sanction Interest',
    'LEASING': 'Leasing Payment'
}

df_transactions['type'] = df_transactions['type'].map(type_map).fillna(df_transactions['type'])
df_transactions['operation'] = df_transactions['operation'].map(operation_map).fillna('Unspecified Operation')
df_transactions['k_symbol'] = df_transactions['k_symbol'].map(k_symbol_map).fillna('Uncategorized/Other')

# 5. Impute missing counterparty attributes for cash transactions
df_transactions['bank'] = df_transactions['bank'].fillna('Internal/Cash')
df_transactions['account'] = df_transactions['account'].fillna(0).astype(int)

# 6. Save the cleaned DataFrame to 'Transaction_clean.csv' in Google Drive
df_transactions.to_csv(clean_trans_path, index=False)

print(f"✅ Successfully cleansed {len(df_transactions):,} transactions and saved to 'Transaction_clean.csv'!")

/tmp/ipykernel_2136/422162649.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_transactions = pd.read_csv(raw_trans_path).dropna(how='all')


✅ Successfully cleansed 1,056,320 transactions and saved to 'Transaction_clean.csv'!


## 2. Demographics & Account Entities (`Client`, `Account`, `Disp`, `Card`)
Standardises client birth dates, computes customer age, parses payment card issuance dates, and maps account statement frequencies.


In [5]:
# ==========================================
# 3. DEMOGRAPHICS & ACCOUNTS CLEANSING
# ==========================================

# A. Clean Client.csv
df_client = pd.read_csv(os.path.join(DATA_DIR, 'Client.csv')).dropna(how='all')
df_client['birth_date'] = pd.to_datetime(df_client['birth_date'], errors='coerce')
df_client['gender'] = df_client['gender'].str.upper().str.strip()

# Calculate customer age relative to dataset end date (1998-12-31)
df_client['age'] = (pd.to_datetime('1998-12-31') - df_client['birth_date']).dt.days // 365
df_client['gender'] = df_client['gender'].str.upper().str.strip()

# B. Clean Account.csv
df_account = pd.read_csv(os.path.join(DATA_DIR, 'Account.csv')).dropna(how='all')
df_account['date'] = pd.to_datetime(df_account['date'], errors='coerce')
freq_map = {
    'POSTATIC MESICNE': 'Monthly Statement',
    'POPLATEK MESICNE': 'Monthly Statement',
    'POPLATEK TYDNE': 'Weekly Statement',
    'POPLATEK PO OBRATU': 'Statement After Transaction'
}
df_account['frequency'] = df_account['frequency'].map(freq_map).fillna(df_account['frequency'])

# C. Clean Disp.csv
df_disp = pd.read_csv(os.path.join(DATA_DIR, 'Disp.csv')).dropna(how='all')
df_disp['type'] = df_disp['type'].str.upper().str.strip()

# D. Clean Card.csv
df_card = pd.read_csv(os.path.join(DATA_DIR, 'Card.csv')).dropna(how='all')
df_card['issued'] = pd.to_datetime(df_card['issued'], errors='coerce')
df_card['type'] = df_card['type'].str.lower().str.strip()

print(f"Client records: {len(df_client):,} | Account records: {len(df_account):,}")
print(f"Disposition records: {len(df_disp):,} | Card records: {len(df_card):,}")


Client records: 5,369 | Account records: 4,500
Disposition records: 5,369 | Card records: 892


## 3. Financial Products & Commitments (`Loan.csv` & `Order.csv`)
Maps loan status codes (`A`, `B`, `C`, `D`) into explicit risk categories, creates binary credit default flags, and cleans recurring payment orders.


In [7]:
# ==========================================
# 4. LOANS & ORDERS CLEANSING
# ==========================================

# A. Clean Loan.csv
df_loan = pd.read_csv(os.path.join(DATA_DIR, 'Loan.csv')).dropna(how='all')
df_loan['date'] = pd.to_datetime(df_loan['date'], errors='coerce')
status_map = {
    'A': 'Finished - Paid in Full',
    'B': 'Finished - Defaulted',
    'C': 'Running - Current/Good',
    'D': 'Running - In Debt/Default'
}
df_loan['status_description'] = df_loan['status'].map(status_map)

# Binary indicator for credit default risk modeling (B and D represent defaulted loans)
df_loan['is_default'] = df_loan['status'].isin(['B', 'D']).astype(int)

# B. Clean Order.csv
df_order = pd.read_csv(os.path.join(DATA_DIR, 'Order.csv')).dropna(how='all')
k_symbol_order_map = {
    'POJISTNE': 'Insurance Payment',
    'SIPO': 'Household/Utility Payment',
    'UVER': 'Loan Payment',
    'LEASING': 'Leasing Payment'
}
df_order['k_symbol'] = df_order['k_symbol'].map(k_symbol_order_map).fillna('Unassigned Payment')

print(f"Loan records: {len(df_loan):,} (Default rate: {df_loan['is_default'].mean():.1%})")
print(f"Order records: {len(df_order):,}")


Loan records: 682 (Default rate: 11.1%)
Order records: 6,471


## 4. District Economic Statistics (`District.csv`)
Renames encoded district feature columns (`A2` through `A16`) into descriptive English metrics and imputes missing regional statistics.


In [6]:
# ==========================================
# 5. DISTRICT DATA CLEANSING & RENAMING
# ==========================================
df_district = pd.read_csv(os.path.join(DATA_DIR, 'District.csv')).dropna(how='all')


# Rename encoded feature columns based on PKDD data dictionary
district_col_map = {
    'A2': 'district_name',
    'A3': 'region',
    'A4': 'inhabitants',
    'A5': 'municipalities_lt_500',
    'A6': 'municipalities_500_1999',
    'A7': 'municipalities_2000_9999',
    'A8': 'municipalities_gt_10000',
    'A9': 'cities_count',
    'A10': 'ratio_urban_inhabitants',
    'A11': 'average_salary',
    'A12': 'unemployment_rate_95',
    'A13': 'unemployment_rate_96',
    'A14': 'entrepreneurs_per_1000',
    'A15': 'crimes_95',
    'A16': 'crimes_96'
}
df_district = df_district.rename(columns=district_col_map)

# Impute missing values in A12 (unemployment_rate_95) and A15 (crimes_95) using regional medians
df_district['unemployment_rate_95'] = df_district['unemployment_rate_95'].fillna(
    df_district.groupby('region')['unemployment_rate_95'].transform('median')
)
df_district['crimes_95'] = df_district['crimes_95'].fillna(
    df_district.groupby('region')['crimes_95'].transform('median')
)

print(f"District records: {len(df_district):,} across {df_district['region'].nunique()} regions.")


District records: 77 across 8 regions.


## 5. Verify loaded datasets summary
To verify all the loaded datasets


In [9]:
datasets = {
    'Transaction': df_transactions,
    'Account': df_account,
    'Card': df_card,
    'Client': df_client,
    'Disp': df_disp,
    'District': df_district,
    'Loan': df_loan,
    'Order': df_order
}

print("\n--- 📊 Dataset Loading Summary ---")
for name, df in datasets.items():
    print(f"• {name:<12}: {len(df):>10,} rows | {df.shape[1]:>2} columns")



--- 📊 Dataset Loading Summary ---
• Transaction :  1,056,320 rows | 10 columns
• Account     :      4,500 rows |  4 columns
• Card        :        892 rows |  4 columns
• Client      :      5,369 rows |  5 columns
• Disp        :      5,369 rows |  4 columns
• District    :         77 rows | 16 columns
• Loan        :        682 rows |  9 columns
• Order       :      6,471 rows |  6 columns


## 6. Export Cleaned Datasets to SQLite Database (`pkdd_capstone.db`)
Exports all cleaned DataFrames directly into an SQLite database for seamless querying, Power BI import, and machine learning model training.


In [10]:
# ==========================================
# 6. EXPORT CLEANED DATASETS
# ==========================================
# Connect to SQLite database
conn = sqlite3.connect('pkdd_capstone.db')

# Write cleaned DataFrames into relational tables
for name, df in datasets.items():
    df.to_sql(name, conn, if_exists='replace', index=False)
conn.close()

print("\n✅ All tables successfully exported to SQLite database ('pkdd_capstone.db')!")



✅ All tables successfully exported to SQLite database ('pkdd_capstone.db')!
